# Übung 06: Datenaufbereitung mit pandas

**Übungsnotebook zur Vorbereitung auf LN 1 (Mo. 09.11.2026)** · Cheatsheet: **Kapitel 6 (Datenaufbereitung mit pandas)** und Kapitel 11 (Checklisten)

| | |
|---|---|
| **Themen** | Auswählen & Filtern, Typumwandlung & Textbereinigung, neue Spalten, fehlende Werte & Duplikate, Ausreisser, groupby/agg, cut/qcut, merge & concat, pivot/melt/crosstab |
| **Format** | wie die Musterprüfungen AP01–AP03: Teil A Konzeptfragen · Teil B Fehleranalyse · Teil C Code schreiben |
| **Zeitbudget** | ca. 250 Minuten (Prüfungstempo: ca. 4.5 Min. pro Punkt) |
| **Lösungen** | keine – alles ist mit dem Cheatsheet bzw. den Kursunterlagen lösbar. Hinweise wie *(→ 4.5)* verweisen auf den Cheatsheet-Abschnitt. |

**Szenario:** Die fiktive Café-Kette *Kafi Kette* hat 5 Filialen in der Region Winterthur. Die Kassendaten von August bis September 2026 wurden aus dem Kassensystem exportiert – mit allen Eigenheiten: uneinheitliche Filialnamen, Preise als Text, zwei Datumsformate, doppelt synchronisierte Bestellungen.

**So üben Sie prüfungsnah**
- Copilot/KI-Autocomplete ausschalten, wie in der Prüfung (Cheatsheet 1.1).
- Begründungen immer **mit konkretem Bezug zu den Daten** (Spaltennamen, Werte, Zahlen aus dem Output) – sonst gibt es in der Prüfung 0 Punkte.
- Verändern Sie die Ausgangsdaten nicht, sondern arbeiten Sie pro Aufgabe mit eigenen Variablen (z.B. `df_B1`, `df_C2`).
- Am Schluss: *Kernel → Restart & Run All* (Cheatsheet 11.5).

## Setup (wird nicht bewertet)

Die folgende Zelle erzeugt die Übungsdaten **synthetisch und reproduzierbar** (fixer Seed) im Unterordner neben diesem Notebook. Sie brauchen den Code nicht zu verstehen und sollen ihn **nicht verändern** – behandeln Sie die erzeugten Dateien so, als hätten Sie sie in der Prüfung erhalten. *(Natürlich dürfen Sie die Dateien im Editor öffnen und ansehen.)*

In [ ]:
# Libraries
import os
import re
import numpy as np
import pandas as pd

# Settings
import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)

# Get current working directory
print(os.getcwd())

In [ ]:
# ---------------------------------------------------------------
# Data generator (do not modify) -> folder 'data_uebung06'
# ---------------------------------------------------------------
import os
import numpy as np
import pandas as pd

rng = np.random.default_rng(606)
D = 'data_uebung06'
os.makedirs(D, exist_ok=True)

stores = pd.DataFrame({'store_id': ['S1', 'S2', 'S3', 'S4', 'S5'],
                       'store': ['Winterthur HB', 'Technikum', 'Töss', 'Oberwinterthur', 'Seuzach'],
                       'seats': [40, 25, 18, 30, 22],
                       'opened': ['2015-03-01', '2018-08-15', '2012-11-01', '2021-05-10', '2024-02-01']})
stores.to_csv(f'{D}/stores.csv', index=False)

products = pd.DataFrame({
    'product_id': [101, 102, 103, 104, 105, 201, 202, 203, 301, 302, 303],
    'name': ['Espresso', 'Cappuccino', 'Latte Macchiato', 'Chai Latte', 'Tee', 'Gipfeli', 'Muffin',
             'Bircher Müesli', 'Sandwich', 'Wrap', 'Salat Bowl'],
    'category': ['Kaffee', 'Kaffee', 'Kaffee', 'Tee & Co', 'Tee & Co', 'Gebäck', 'Gebäck', 'Frühstück',
                 'Lunch', 'Lunch', 'Lunch'],
    'price_chf': [4.20, 5.40, 5.80, 5.90, 4.50, 2.20, 4.20, 6.50, 8.90, 9.50, 13.90]})
popularity = np.array([14, 20, 10, 4, 5, 12, 6, 4, 8, 5, 4], dtype=float)
popularity /= popularity.sum()

# product master with history: price change on 2026-09-01 for some products (2 rows per product_id!)
p_old = products.copy(); p_old['valid_from'] = '2025-01-01'
p_new = products[products['product_id'].isin([102, 103, 201, 301, 303])].copy()
p_new['price_chf'] = (p_new['price_chf'] + 0.30).round(2); p_new['valid_from'] = '2026-09-01'
pd.concat([p_old, p_new]).sort_values(['product_id', 'valid_from']).to_csv(f'{D}/products.csv', index=False)

store_names = {'Winterthur HB': ['Winterthur HB', 'winterthur hb', ' Winterthur HB', 'Winterthur Hb'],
               'Technikum': ['Technikum', 'technikum', 'Technikum '],
               'Töss': ['Töss', 'TÖSS', 'Toess'],
               'Oberwinterthur': ['Oberwinterthur', 'oberwinterthur'],
               'Seuzach': ['Seuzach', 'seuzach ']}
store_w = np.array([0.32, 0.22, 0.14, 0.18, 0.14])

n = 6000
rows = []
for i in range(n):
    st = stores['store'].iloc[rng.choice(5, p=store_w)]
    pi = rng.choice(len(products), p=popularity)
    pid = int(products['product_id'].iloc[pi])
    ts = pd.Timestamp('2026-08-17') + pd.Timedelta(days=int(rng.integers(0, 42)),
                                                  hours=int(rng.choice(np.arange(6, 19), p=np.array([2, 8, 10, 7, 5, 6, 9, 9, 5, 4, 4, 3, 2.]) / 74)),
                                                  minutes=int(rng.integers(0, 60)), seconds=int(rng.integers(0, 60)))
    if ts.weekday() == 6 and st != 'Winterthur HB':          # only the main station store opens on Sunday
        ts -= pd.Timedelta(days=1)
    price = float(products['price_chf'].iloc[pi]) + (0.30 if (pid in (102, 103, 201, 301, 303) and ts >= pd.Timestamp('2026-09-01')) else 0)
    qty = int(rng.choice([1, 1, 1, 1, 1, 2, 2, 3]))
    loyalty = f'L{int(rng.integers(1000, 1600))}' if (rng.random() < 0.45 and st != 'Töss') else None
    if st == 'Töss' and ts >= pd.Timestamp('2026-09-14') and rng.random() < 0.45:
        loyalty = f'L{int(rng.integers(1000, 1600))}'        # new terminal records loyalty cards
    rows.append({'order_id': f'O{100000 + i}', 'timestamp': ts, 'store_true': st, 'product_id': pid,
                 'qty': qty, 'unit_price': round(price, 2), 'loyalty_id': loyalty,
                 'payment': str(rng.choice(['Karte', 'TWINT', 'Bar'], p=[0.5, 0.35, 0.15]))})
o = pd.DataFrame(rows)

# returns and a test booking
ret = rng.choice(n, size=25, replace=False)
o.loc[ret, 'qty'] = -1
o.loc[1234, 'qty'] = 999
o.loc[1234, ['store_true', 'product_id', 'unit_price']] = ['Seuzach', 101, 4.20]

# messy text formats
o['store'] = [rng.choice(store_names[s]) for s in o['store_true']]
def price_text(p, i):
    v = i % 4
    if v == 0:
        return f'{p:.2f}'
    if v == 1:
        return f'CHF {p:.2f}'
    if v == 2:
        return f'{p:.2f}'.replace('.', ',')
    return f'{p:.2f}'
o['unit_price_raw'] = [price_text(p, i) for i, p in enumerate(o['unit_price'])]
o.loc[4321, 'unit_price_raw'] = '450'                       # typo at the till: 4.50 -> decimal point lost
o.loc[4321, ['store', 'store_true', 'product_id', 'unit_price', 'qty']] = ['Technikum', 'Technikum', 105, 4.50, 2]
o['ts_text'] = [t.strftime('%d.%m.%Y %H:%M') if (k % 9 == 0) else t.strftime('%Y-%m-%d %H:%M:%S')
                for k, t in enumerate(o['timestamp'])]
o['synced_at'] = (o['timestamp'] + pd.to_timedelta(rng.integers(1, 300, n), unit='s')).dt.strftime('%Y-%m-%d %H:%M:%S')

# duplicates from a POS sync problem: same order, new sync timestamp
dup = o.sample(60, random_state=6).copy()
dup['synced_at'] = (pd.to_datetime(dup['synced_at']) + pd.Timedelta(hours=2)).dt.strftime('%Y-%m-%d %H:%M:%S')
o_all = pd.concat([o, dup]).sort_values('timestamp').reset_index(drop=True)

out = o_all[['order_id', 'ts_text', 'synced_at', 'store', 'product_id', 'qty', 'unit_price_raw', 'loyalty_id', 'payment']]
out = out.rename(columns={'ts_text': 'timestamp', 'unit_price_raw': 'unit_price'})
out.to_csv(f'{D}/orders.csv', index=False)

sept = o[(o['timestamp'] >= '2026-09-01') & (o['timestamp'] < '2026-10-01')]
print('Files created:', sorted(os.listdir(D)))
print('orders.csv:', len(out), 'Zeilen')
print('Kontrollwert Kassensystem: Anzahl Bestellungen im September 2026 =', sept['order_id'].nunique())

### Daten laden

| Datei | Spalten | Hinweise |
|---|---|---|
| `orders.csv` | `order_id`, `timestamp`, `synced_at`, `store`, `product_id`, `qty`, `unit_price`, `loyalty_id`, `payment` | eine Zeile = eine Bestellung mit **einem** Produkt; `qty = -1` sind Retouren; `loyalty_id` = Kundenkarte (falls vorgezeigt) |
| `products.csv` | `product_id`, `name`, `category`, `price_chf`, `valid_from` | Produktstamm **mit Preis-Historie** |
| `stores.csv` | `store_id`, `store`, `seats`, `opened` | die 5 Filialen mit korrekter Schreibweise |

In [ ]:
# Load the data (do not modify df_orders, df_products, df_stores)
df_orders = pd.read_csv('data_uebung06/orders.csv')
df_products = pd.read_csv('data_uebung06/products.csv')
df_stores = pd.read_csv('data_uebung06/stores.csv')

for name, d in [('orders', df_orders), ('products', df_products), ('stores', df_stores)]:
    print(f'{name:<9}', d.shape)
df_orders.head()

> **Hinweis:** Alle Aufgaben sind voneinander unabhängig. Arbeiten Sie pro Aufgabe mit einer Kopie (z.B. `df_C1 = df_orders.copy()`).

---

<div style="background-color:#2E4057; color:white; padding:16px 24px; border-radius:6px; font-size:18px; margin-top:8px;">
&#9654;&nbsp; <strong>Ab hier beginnen die Übungsaufgaben — 60 Punkte</strong><br>
<span style="font-size:15px; font-weight:normal;">Tragen Sie Ihre Antworten direkt in die vorgesehenen Zellen ein und führen Sie alle Code-Zellen aus.</span>
</div>

---
## Teil A – Konzeptfragen (12 Punkte)

*Antworten ohne Bezug zu den Daten der Kafi Kette werden nicht gewertet.*

### A1 (4 Punkte)

In `orders.csv` finden Sie u.a. eine Bestellung mit `unit_price = 450`, eine mit `qty = 999` und 25 Bestellungen mit `qty = -1`. Ermitteln Sie die betroffenen Zeilen mit geeignetem Code (z.B. `nlargest`, Filter).

Entscheiden Sie für **jeden** der drei Fälle, ob es sich um einen Datenfehler oder einen echten Wert handelt, und wie Sie damit umgehen (korrigieren, entfernen, separat ausweisen). Begründen Sie mit den Daten (z.B. Produkt, Filiale, Uhrzeit). *(→ 6.5)*

In [ ]:
# Code zur Untersuchung hier eintragen

**Ihre Antwort:**

> [Hier eintragen]

### A2 (4 Punkte)

Die Spalte `loyalty_id` ist bei vielen Bestellungen leer. Untersuchen Sie den Anteil fehlender Werte **pro Filiale** (und für die Filiale Töss zusätzlich **vor und ab dem 14.09.2026**).

a) Welcher Mechanismus (MCAR, MAR, MNAR) liegt vor? Begründen Sie mit Ihren Zahlen.

b) Das Marketing berechnet «Anteil Kundenkarten-Bestellungen pro Filiale» und folgert, Töss habe die treueste Kundschaft bzw. die schwächste. Was ist an dieser Auswertung problematisch, und wie würden Sie sie korrekt durchführen? *(→ 6.4, 9.4)*

In [ ]:
# Code zur Untersuchung hier eintragen

**Ihre Antwort:**

> [Hier eintragen]

### A3 (4 Punkte)

Der Bestellwert (`qty × Preis`) soll in drei Klassen eingeteilt werden. Variante 1: `pd.cut(..., bins=3)`; Variante 2: `pd.qcut(..., q=3)`.

Erklären Sie den Unterschied am Beispiel dieser Daten (rechtsschiefe Bestellwerte, wenige grosse Lunch-Bestellungen): Welche Variante führt zu fast leeren Klassen, welche zu ungefähr gleich grossen? Was passiert jeweils mit Bestellungen ohne gültigen Preis (`NaN`)? *(→ 6.7)*

**Ihre Antwort:**

> [Hier eintragen]

---
## Teil B – Fehleranalyse Python-Code (24 Punkte)

*Jeder Code läuft fehlerfrei durch, liefert aber ein falsches oder irreführendes Ergebnis.*

### B1 (8 Punkte)

**Ziel:** Anzahl Bestellungen und verkaufte Menge (`qty`) pro **Produktkategorie** – auf Basis aller Zeilen in `orders.csv` (Duplikate und Retouren sind hier nicht das Thema). Jede Zeile in `orders.csv` enthält **genau ein** Produkt.

In [ ]:
# Original code (do not modify this cell)
df_B1_orig = df_orders.merge(df_products, on='product_id', how='left')

print('Zeilen orders:', len(df_orders), '| Zeilen nach merge:', len(df_B1_orig))
print(df_B1_orig.groupby('category').agg(n_orders=('order_id', 'count'), qty=('qty', 'sum'))
                .sort_values('n_orders', ascending=False))

**Aufgabe:** Erklären Sie, warum die Zeilenzahl nach dem `merge` steigt und welche Kategorien dadurch überschätzt werden. Schreiben Sie den korrigierten Code. Stellen Sie mit einem Parameter von `merge()` sicher, dass ein solcher Fehler künftig **sofort** auffällt. Begründen Sie jede Änderung. *(→ 6.8, 11.3)*

**Begründung der Änderungen:**

> [Hier eintragen]

**Korrigierter Code:**

In [ ]:
# Korrigierten Code unter den Kommentaren eintragen

### B2 (8 Punkte)

**Ziel:** Rangliste der Filialen nach **typischem Bestellwert** (`qty × Preis`). Die Kette hat **5 Filialen** (siehe `stores.csv`). Die Leiterin der Filiale *Seuzach* (jüngste Filiale, vor allem Kaffee to go für Pendler) wundert sich, dass ihre Filiale mit grossem Abstand ganz oben steht.

In [ ]:
# Original code (do not modify this cell)
df_B2_orig = df_orders.copy()
df_B2_orig['price'] = pd.to_numeric(df_B2_orig['unit_price'].str.replace('CHF ', '')
                                                            .str.replace(',', '.'))
df_B2_orig['order_value'] = df_B2_orig['qty'] * df_B2_orig['price']

df_B2_orig.groupby('store')['order_value'].mean().round(2).sort_values(ascending=False)

**Aufgabe:** Untersuchen Sie mit geeigneten Methoden (z.B. `value_counts()`, `describe()`, `nlargest()`), warum die Rangliste irreführend ist. Schreiben Sie den korrigierten Code, der eine Rangliste mit **genau 5 Filialen** und **robuster** Kennzahl liefert (inkl. Anzahl Bestellungen pro Filiale). Begründen Sie jede Änderung. *(→ 6.2, 6.5, 6.6, 11.3)*

**Begründung der Änderungen:**

> [Hier eintragen]

**Korrigierter Code:**

In [ ]:
# Korrigierten Code unter den Kommentaren eintragen

### B3 (8 Punkte)

**Ziel:** Anzahl Bestellungen im **September 2026**. Der **Kontrollwert des Kassensystems** wird im Setup ausgegeben. `order_id` ist pro Bestellung eindeutig.

In [ ]:
# Original code (do not modify this cell)
df_B3_orig = df_orders.drop_duplicates()
df_B3_orig['ts'] = pd.to_datetime(df_B3_orig['timestamp'], format='%Y-%m-%d %H:%M:%S', errors='coerce')

sept = df_B3_orig[(df_B3_orig['ts'] >= '2026-09-01') & (df_B3_orig['ts'] < '2026-10-01')]
print('Duplikate entfernt:', len(df_orders) - len(df_B3_orig))
print('Bestellungen September:', len(sept))

**Aufgabe:** Finden Sie **beide** Ursachen für die Abweichung vom Kontrollwert. Schreiben Sie den korrigierten Code, der den Kontrollwert exakt trifft, und begründen Sie jede Änderung. Wie hätten Sie die beiden Probleme mit einer kurzen Kontrolle nach jedem Schritt entdecken können? *(→ 2.6, 6.2, 6.4, 1.3)*

**Begründung der Änderungen:**

> [Hier eintragen]

**Korrigierter Code:**

In [ ]:
# Korrigierten Code unter den Kommentaren eintragen

---
## Teil C – Python-Code schreiben (24 Punkte)

*Schreiben Sie Python-Code und begründen Sie Ihre Vorgehensweise jeweils in 1–2 Sätzen.*

### C1 (8 Punkte)

Erstellen Sie einen bereinigten Data Frame `df_clean` aus `df_orders` mit folgenden Schritten. Kontrollieren Sie nach **jedem** Schritt kurz das Ergebnis (`shape`, `isna().sum()`, `value_counts()`, `describe()`).

1. Duplikate (gleiche `order_id`) entfernen – die **zuerst** synchronisierte Zeile behalten.
2. `store` auf die 5 offiziellen Namen aus `stores.csv` vereinheitlichen.
3. `ts` als `datetime` aus beiden vorkommenden Formaten erstellen (keine `NaT`).
4. `price` als `float` aus `unit_price` (alle Formate); den Tippfehler `450` korrigieren (Begründung!).
5. Die Testbuchung (`qty = 999`) entfernen; Retouren behalten und mit einer Spalte `is_return` (bool) kennzeichnen.
6. Neue Spalten: `order_value = qty × price`, `weekday` (0 = Montag), `hour`.

Geben Sie zum Schluss `shape`, `dtypes` und `describe()` von `df_clean` aus. Begründen Sie Ihre Entscheidungen bei Schritt 1, 4 und 5. *(→ 6.1–6.5, 2.6)*

**Begründung der Vorgehensweise:**

> [Hier eintragen]

In [ ]:
# Code hier eintragen

### C2 (4 Punkte)

Erstellen Sie mit `df_clean` (aus C1) pro Filiale eine Kennzahlentabelle mit **benannter Aggregation**: `n_orders`, `revenue_chf` (Summe `order_value`), `median_value`, `return_rate_pct` (Anteil Retouren in %, 1 Stelle) und `seats` (aus `stores.csv` per `merge`). Berechnen Sie zusätzlich `revenue_per_seat`. Runden Sie Beträge auf 2 Stellen und sortieren Sie absteigend nach `revenue_per_seat`. Begründen Sie die Wahl des Join-Typs. *(→ 6.6, 6.8)*

**Begründung der Vorgehensweise:**

> [Hier eintragen]

In [ ]:
# Code hier eintragen

### C3 (4 Punkte)

Erstellen Sie mit `df_clean` eine Tabelle **Wochentag × Stunde** mit der Anzahl Bestellungen (`pivot_table`, fehlende Kombinationen = 0). Die Zeilen sollen mit `Mo` … `So` beschriftet und in dieser Reihenfolge sortiert sein. Formen Sie die Tabelle anschliessend mit `melt` wieder ins Long-Format um und geben Sie die **5 stärksten** Zeitfenster (Wochentag + Stunde) aus. Begründen Sie, wofür Long- bzw. Wide-Format jeweils geeignet ist. *(→ 6.9)*

**Begründung der Vorgehensweise:**

> [Hier eintragen]

In [ ]:
# Code hier eintragen

### C4 (4 Punkte)

Verwenden Sie nur Bestellungen **mit** Kundenkarte aus `df_clean`. Berechnen Sie pro `loyalty_id` den Gesamtumsatz und teilen Sie die Karteninhaber mit `pd.qcut` in **4 gleich grosse** Gruppen mit den Labels `Q1` (tiefster Umsatz) bis `Q4`. Prüfen Sie die Gruppengrössen und die Klassengrenzen (`retbins=True`).

Erstellen Sie anschliessend eine Kreuztabelle **Umsatzgruppe × Kategorie** (Anzahl Bestellungen) mit **Zeilenanteilen in Prozent** (1 Stelle). Interpretieren Sie in 1–2 Sätzen, worin sich Q4- von Q1-Kunden unterscheiden. *(→ 6.6–6.9)*

**Begründung und Interpretation:**

> [Hier eintragen]

In [ ]:
# Code hier eintragen

### C5 (4 Punkte)

Bilden Sie mit `np.select` eine Spalte `daypart`: `Morgen` (vor 11 Uhr), `Mittag` (11 bis vor 14 Uhr), `Nachmittag` (ab 14 Uhr). Erstellen Sie eine Kreuztabelle **Filiale × Tageszeit** mit Zeilenanteilen in Prozent (1 Stelle) und einer Zeile `Total` (`margins=True`). Welche Filiale ist am stärksten vom Morgengeschäft abhängig? Begründen Sie mit den Zahlen. *(→ 6.3, 6.9)*

**Begründung und Interpretation:**

> [Hier eintragen]

In [ ]:
# Code hier eintragen

---
### Jupyter notebook --footer info-- (please always provide this at the end of each notebook)

In [ ]:
# Libraries
import os, platform, socket
from platform import python_version
from datetime import datetime

# Print system information
print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime('%Y-%m-%d %H:%M:%S'))
print('Python Version:', python_version())
print('-----------------------------------')